[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m01-workflow-refresher/lab/lab1_workflow_refresher.ipynb)

# Lab 1 — Reproducible workflow, statistics refresher and SQL

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import sys
import numpy as np
import pandas as pd

**Course:** 040563107 Tools for Statistical Data Science · **Module m01** · **Time:** 3 hours · **Practice lab** (not graded; use the self-checks)

**After this lab you can:**
* run a course notebook in Colab and read its environment check (objective m01-O1);
* repair a notebook that only works because of hidden state, so that it passes *Restart and run all* (O2);
* load a dataset through the course registry and report its licence and citation (O3);
* choose between arithmetic, geometric and harmonic means, and compute a sample standard deviation with the right `ddof` (O4);
* detect and recode a sentinel missing-value code before summarising (O5);
* write SQL aggregations and window functions in DuckDB that reproduce the pandas results (O6);
* audit an AI-generated code cell: find the error and prove the fix with a test (O7).

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before you finish, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors.
5. Fill in the **AI-use log** at the end.

This lab is practice: it is not graded, but it uses exactly the format of the six graded labs, so it is also your rehearsal for them.

## Part 0 — Your environment (about 10 minutes)

The setup cell installed the course helper `tsds` (if needed) and printed a table of package versions.
Two people can run the *same* notebook and get *different* results when their package versions differ;
the course tests every lab against two pinned environments so that this does not happen silently.

**Q1.** Copy the **Runtime** line and the **pandas** version that the setup cell printed. Give one concrete reason why two students
running the same notebook could get different numbers, and name one thing in this course's setup that prevents it.

*Your answer:* 

## Part 1 — Hidden state (about 30 minutes)

A notebook remembers every variable you created, **in the order you ran the cells**, not the order they appear.
A notebook that "works on my laptop" may only work because its author ran the cells out of order, or ran one twice.

`run_cells` below imitates a notebook session: it starts from a **fresh** namespace (like *Restart*) and runs the cells
in the order you give. `run_cells(cells)` is *Restart and run all*; `run_cells(cells, [0, 2, 2, 1])` is what happens when
you run cell 0, then run cell 2 **twice** (for example after editing a comment in it), then cell 1.

The author of `cells_broken` ran the cells in the order 0, 2, 1 and got `summary == {"mean": 0.63, "sd": 0.1512}`.

In [ ]:
def run_cells(cells, order=None):
    """Run cell sources in a fresh namespace, in `order` (default: top to bottom); return the namespace."""
    ns = {}
    for i in (range(len(cells)) if order is None else order):
        try:
            exec(cells[i], ns)
        except Exception as e:
            raise RuntimeError(f"cell {i} failed: {type(e).__name__}: {e}") from None
    return ns


cells_broken = [
    "import numpy as np\nraw = np.array([52.0, 61.0, 70.0, 85.0, 47.0])  # exam scores in percent",
    "summary = {'mean': round(float(prop.mean()), 4), 'sd': round(float(prop.std(ddof=1)), 4)}",
    "raw = raw / 100  # percent -> proportion\nprop = raw",
]

for order in ([0, 2, 1], [0, 1, 2], [0, 2, 2, 1]):
    try:
        print(order, "->", run_cells(cells_broken, order)["summary"])
    except RuntimeError as e:
        print(order, "->", e)

In [ ]:
# Task T1
# Write `fixed_cells`: a list of three cell sources such that
#   * Restart and run all works: run_cells(fixed_cells) raises no error and gives
#     summary == {"mean": 0.63, "sd": 0.1512};
#   * every cell is safe to run again: running each cell twice in a row, run_cells(fixed_cells, [0, 0, 1, 1, 2, 2]),
#     gives the same summary.
# Change as little as possible.
# YOUR CODE HERE
raise NotImplementedError()
print(run_cells(fixed_cells)["summary"])

In [ ]:
# Self-check for task T1: run it; it should print OK.
assert len(fixed_cells) == 3, "keep three cells"
once = run_cells(fixed_cells)["summary"]
twice = run_cells(fixed_cells, [0, 0, 1, 1, 2, 2])["summary"]
assert np.isclose(once["mean"], 0.63) and np.isclose(once["sd"], 0.1512), f"summary is {once}"
assert once == twice, f"running each cell twice changed the result: {once} -> {twice} (a cell overwrites its own input)"
print("OK")

**Q2.** Name the **two** different problems in `cells_broken` (one shows when the notebook is run top to bottom, the other only when a cell
is run again). For each, say whether *Restart and run all* reveals it, and what fixed it.

*Your answer:* 

## Part 2 — Which mean? Which standard deviation? (about 30 minutes)

Three facts to check before you summarise anything:
* **Floating point:** `0.1 + 0.2` is not exactly `0.3`; compare decimals with `np.isclose`, not `==`.
* **Means:** the arithmetic mean answers "what is the average *amount*?"; the geometric mean answers "what *constant growth rate*
  gives the same total?"; the harmonic mean answers "what *constant rate* covers the same distance in the same time?".
* **`ddof`:** numpy's `np.std` divides by $n$ by default; pandas' `.std()` divides by $n-1$ (the sample standard deviation).

In [ ]:
print("0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3, "| np.isclose:", np.isclose(0.1 + 0.2, 0.3))
x = np.array([52.0, 61.0, 70.0, 85.0, 47.0])
print("np.std(x):", np.std(x).round(4), "| pd.Series(x).std():", pd.Series(x).std().round(4))

returns = np.array([0.10, -0.05, 0.20, -0.15, 0.08])  # monthly returns of an investment
speeds = np.array([30.0, 60.0, 90.0])                 # km/h on three legs of 60 km each

In [ ]:
# Task T2
# (a) am_return: arithmetic mean of `returns`; gm_return: the constant monthly return that gives the same total growth
#     (geometric mean of the growth factors 1 + r, minus 1).
# (b) hm_speed: the average speed over the whole 180 km trip.
# (c) sd_sample: the sample standard deviation of x, computed from the formula (no np.std, no .std()).
# YOUR CODE HERE
raise NotImplementedError()
print(f"AM return {am_return:.4f} | GM return {gm_return:.4f} | average speed {hm_speed:.2f} km/h | sd {sd_sample:.4f}")

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert np.isclose((1 + gm_return) ** len(returns), np.prod(1 + returns)), "compounding gm_return must reproduce the total growth"
assert np.isclose(am_return, returns.mean())
assert np.isclose(hm_speed, 180 / np.sum(60 / speeds)), "average speed = total distance / total time"
assert np.isclose(sd_sample, pd.Series(x).std()), "use the n - 1 divisor"
print("OK")

**Q3.** An advisor reports "the investment returned 3.6% a month on average". Is that the right summary for an investor? Use your
`am_return` and `gm_return`, and say which mean you would report and why. Then say in one sentence why `hm_speed` is below 60 km/h.

*Your answer:* 

## Part 3 — Provenance and a sentinel code (about 40 minutes)

Every dataset in this course comes through the registry, with a licence and a citation that you must give when you use it.
The Air Quality data are hourly readings from a gas sensor device in an Italian city (March 2004 – April 2005).
`tsds.data.load` downloads the file from UCI, checks its SHA-256 checksum, and returns it **raw**.

In [ ]:
meta = tsds.data.info("air_quality")
print("Notes:", meta["notes"])
raw = tsds.data.load("air_quality")
print(raw.shape)
raw.head()

In [ ]:
raw.describe().T[["count", "mean", "min", "max"]].round(2)

Look at the `min` column above. A minimum temperature of −200 °C is not a measurement: the data documentation says missing
values are coded as **−200**. Statistics computed before recoding are wrong, and nothing raises an error.

Also notice the trailing empty rows and the two empty `Unnamed` columns (the file ends every line with `;;`).
Use these short column names:

In [ ]:
NAMES = {"CO(GT)": "co", "PT08.S1(CO)": "s1_co", "NMHC(GT)": "nmhc", "C6H6(GT)": "benzene", "PT08.S2(NMHC)": "s2_nmhc",
         "NOx(GT)": "nox", "PT08.S3(NOx)": "s3_nox", "NO2(GT)": "no2", "PT08.S4(NO2)": "s4_no2", "PT08.S5(O3)": "s5_o3",
         "T": "temp", "RH": "rh", "AH": "ah"}

In [ ]:
# Task T3
# Build `aq` from `raw`: one row per hour, with
#   * the empty rows and the empty "Unnamed" columns removed,
#   * a DatetimeIndex named "ts" made from Date and Time (format "%d/%m/%Y %H.%M.%S"); then drop Date and Time,
#   * the columns renamed with NAMES,
#   * the sentinel -200 replaced by NaN everywhere.
# Do not modify `raw` itself (build a new object).
# YOUR CODE HERE
raise NotImplementedError()
print(aq.shape, aq.index.min(), "to", aq.index.max())

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert isinstance(aq.index, pd.DatetimeIndex) and aq.index.name == "ts", "index must be a DatetimeIndex named ts"
assert list(aq.columns) == list(NAMES.values()), "columns: the 13 measurements, renamed with NAMES, in the original order"
assert aq.index.is_monotonic_increasing and aq.index.is_unique
assert (aq.index.to_series().diff().dropna() == pd.Timedelta("1h")).all(), "rows should be consecutive hours"
assert not (aq == -200).any().any(), "a -200 sentinel is still in the data"
assert aq["temp"].isna().sum() == (raw["T"] == -200).sum(), "every -200 temperature should be NaN"
assert (raw == -200).any().any(), "do not modify raw"
print("OK")

In [ ]:
# Task T4
# (a) missing_share: Series with the share of missing values in each column of aq, sorted from most to least missing.
# (b) temp_means: dict {"raw": mean of raw["T"], "clean": mean of aq["temp"]}.
# YOUR CODE HERE
raise NotImplementedError()
print(missing_share.round(3).head(5))
print({k: round(v, 2) for k, v in temp_means.items()})

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert set(missing_share.index) == set(aq.columns) and missing_share.is_monotonic_decreasing
assert np.isclose(missing_share["co"], aq["co"].isna().mean())
assert temp_means["clean"] - temp_means["raw"] > 5, "the clean mean should be far above the raw one"
print("OK")

**Q4.** **Failure analysis.** Using `temp_means` and `missing_share`: (a) how wrong is the mean temperature if the sentinel is treated as data,
and why did no error warn you? (b) Would you use the column `nmhc` in an analysis? Give a reason based on the numbers.

*Your answer:* 

## Part 4 — The same answer in SQL (about 40 minutes)

DuckDB runs SQL directly on a pandas DataFrame, on your laptop or in Colab, with no server. Writing the same summary two ways
(pandas and SQL) and checking that they agree is a cheap, strong test of both.

Two things to know:
* SQL aggregates (`AVG`, `COUNT(col)`) **skip NULL**; pandas' `NaN` becomes SQL `NULL`.
* A window function computes a value for every row from a "window" of neighbouring rows, e.g.
  `AVG(co) OVER (ORDER BY ts ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)` is a 24-row trailing mean.

In [ ]:
try:
    import duckdb
except ImportError:  # not preinstalled in this runtime
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
print("duckdb", duckdb.__version__)
print(duckdb.sql("SELECT 42 AS answer").df())

In [ ]:
# Task T5
con = duckdb.connect()
con.register("aq_tbl", aq.reset_index())  # the table has a column ts plus the 13 measurements

# (a) daily_pd: DataFrame with one row per calendar day (sorted), columns co_mean (mean CO over valid hours) and
#     n_valid (number of hours with a valid CO reading). Use pandas.
# (b) daily_sql: the same from SQL on aq_tbl, columns day, co_mean, n_valid, ordered by day.
#     Hint: CAST(ts AS DATE), GROUP BY, COUNT(co).
# YOUR CODE HERE
raise NotImplementedError()
print(len(daily_pd), "days;", int((daily_pd["n_valid"] == 0).sum()), "days without any valid CO reading")
daily_sql.head()

In [ ]:
# Self-check for task T5: run it; it should print OK.
assert list(daily_sql.columns) == ["day", "co_mean", "n_valid"]
assert len(daily_pd) == len(daily_sql) == aq.index.normalize().nunique(), "one row per calendar day"
assert np.allclose(daily_pd["co_mean"].to_numpy(), daily_sql["co_mean"].to_numpy(), equal_nan=True), "pandas and SQL disagree"
assert (daily_pd["n_valid"].to_numpy() == daily_sql["n_valid"].to_numpy()).all()
assert daily_pd["n_valid"].sum() == aq["co"].notna().sum()
print("OK")

In [ ]:
# Task T6
# 24-hour trailing mean of CO (the current hour and the 23 before it), ignoring missing hours.
# (a) co24_pd: from pandas (.rolling). (b) co24_sql: a Series/array from a SQL window function on aq_tbl, in ts order.
# The two must agree everywhere, including where they are NaN.
# YOUR CODE HERE
raise NotImplementedError()
print(pd.DataFrame({"co": aq["co"], "pandas": co24_pd, "sql": np.asarray(co24_sql)}).iloc[20:26].round(3))

In [ ]:
# Self-check for task T6: run it; it should print OK.
a, b = np.asarray(co24_pd, dtype=float), np.asarray(co24_sql, dtype=float)
assert a.shape == b.shape == (len(aq),)
assert np.array_equal(np.isnan(a), np.isnan(b)), \
    "NaN in different places: pandas' rolling needs min_periods=1 to skip missing hours like SQL's AVG does"
assert np.allclose(a, b, equal_nan=True)
print("OK")

A window of 24 **rows** is a window of 24 **hours** only because the T3 self-check confirmed there are no gaps in the hourly
index. With gaps you would need a time-based window (`.rolling("24h")` or `RANGE BETWEEN INTERVAL 23 HOURS PRECEDING AND CURRENT ROW`).

## Part 5 — Audit an AI answer (about 25 minutes)

AI assistants are allowed in this course **with disclosure**, and you are responsible for every cell you submit. The skill is not
"spotting that something looks odd" but **proving** it with a test that you could have written before seeing the answer.

In [ ]:
# Task T7
# An AI assistant was asked: "Using aq, compute the daily mean CO concentration." It answered with the next line.
# Do not edit it.
daily_ai = aq.groupby(aq.index.day)["co"].mean()
print("AI answer:", len(daily_ai), "values; first three:", daily_ai.head(3).round(3).to_dict())

# (a) Write test_daily(daily): it must raise AssertionError for a wrong daily series and pass for a correct one.
#     Base it on properties you can state in advance (e.g. how many days the data cover, or the value for one day
#     computed by hand), not on the AI's numbers.
# (b) daily_fixed: the corrected Series of daily mean CO (one value per calendar day, in date order).
def test_daily(daily):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
test_daily(daily_fixed)
print("daily_fixed passes:", len(daily_fixed), "values")

In [ ]:
# Self-check for task T7: run it; it should print OK.
try:
    test_daily(daily_ai)
    rejected = False
except AssertionError:
    rejected = True
assert rejected, "test_daily must raise AssertionError for the AI answer"
test_daily(daily_fixed)
assert np.allclose(np.asarray(daily_fixed, dtype=float), daily_pd["co_mean"].to_numpy(), equal_nan=True)
print("OK")

**Q5.** **AI audit.** What exactly does the AI's line compute, and why does its output look plausible at a glance? Which property in your
`test_daily` exposes it? State one habit you will use to check AI-written code in this course.

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *the DuckDB syntax for a window function* | *used the OVER clause; wrote the query myself* | *T6 self-check: SQL and pandas agree* |
| | | | |

If you used no AI tool, write "None".

**Cite the data** when you use them: Vito, S. (2008). Air Quality [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C59K5F (CC BY 4.0).